In [6]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess 
import numpy as np 
import math 
import pandas as pd
import chess 
import chess.pgn
import chess
import chess.pgn
import os
import sys
from collections import Counter
from tqdm import tqdm
import pickle

In [ ]:
# --- 1. Clean Token Logic ---
def get_compact_token(uci_str, piece_char):
    """
    Format: {from_sq}{piece}{dist}{direction}
    Examples: e2P2, g1N3, c1B4NE, e4P1NE
    """
    # Parse Coordinates from UCI (e.g., "e2e4")
    from_sq_str = uci_str[:2]
    to_sq_str = uci_str[2:4]
    promo = uci_str[4:].upper() if len(uci_str) > 4 else ""

    # Math: file (0-7), rank (0-7)
    f_file, f_rank = ord(from_sq_str[0]) - ord('a'), int(from_sq_str[1]) - 1
    t_file, t_rank = ord(to_sq_str[0]) - ord('a'), int(to_sq_str[1]) - 1
    
    d_file = t_file - f_file
    d_rank = t_rank - f_rank

    # 1. Castling (Explicit)
    if piece_char == 'K' and abs(d_file) == 2:
        return f"{from_sq_str}CK" if d_file > 0 else f"{from_sq_str}CQ"

    # 2. Promotion (Explicit)
    if promo:
        return f"{from_sq_str}P{promo}" # e.g. "a7PQ"

    # 3. Geometry Calculation
    dist = max(abs(d_file), abs(d_rank))
    
    if dist > 0:
        step_f = d_file // dist
        step_r = d_rank // dist
    else:
        step_f, step_r = 0, 0

    dir_map = {
        (0, 1): "N",  (1, 1): "NE",   (1, 0): "E",  (1, -1): "SE",
        (0, -1): "S", (-1, -1): "SW", (-1, 0): "W", (-1, 1): "NW"
    }
    
    # 4. Knights (Specific Indices 1-8)
    if piece_char == 'N':
        knight_map = {
            (1, 2): "1", (2, 1): "2", (2, -1): "3", (1, -2): "4",
            (-1, -2): "5", (-2, -1): "6", (-2, 1): "7", (-1, 2): "8"
        }
        # Result: g1N3 (Clean!)
        return f"{from_sq_str}N{knight_map.get((d_file, d_rank), '')}"

    # 5. Pawns
    dir_code = dir_map.get((step_f, step_r), "")
    if piece_char == 'P':
        # If moving straight North (Push), omit direction -> e2P2
        if dir_code == "N": 
            return f"{from_sq_str}P{dist}"
        # Captures/EP have diag direction -> e4P1NE
        return f"{from_sq_str}P{dist}{dir_code}"

    # 6. Sliders (B, R, Q, K) - Keep specific types!
    # e.g. c1B4NE, d1Q7N
    return f"{from_sq_str}{piece_char}{dist}{dir_code}"


# --- 2. Mirroring Helper (Strings only) ---
def mirror_uci_string(uci):
    # 'e2e4' -> 'e7e5' (flipped ranks)
    new_from = str(9 - int(uci[1]))
    new_to = str(9 - int(uci[3]))
    return f"{uci[0]}{new_from}{uci[2]}{new_to}{uci[4:]}"


# --- 3. Main Processor ---
def process_pgn(pgn_path, output_path):
    all_tokens = Counter()
    
    # Open files safely
    with open(pgn_path) as pgn_f, open(output_path, 'w') as out_f:
        pbar = tqdm(desc="Games Processed", unit=" games")
        
        while True:
            try:
                game = chess.pgn.read_game(pgn_f)
            except ValueError:
                continue
                
            if game is None: break
            
            # --- Result Filtering ---
            result = game.headers.get("Result", "*")
            should_mirror = (result == "0-1")
            
            board = game.board()
            game_tokens = []
            
            for node in game.mainline():
                move = node.move
                
                # A. Get Info from REAL board
                piece = board.piece_at(move.from_square)
                # Fallback if PGN is malformed and piece is None
                if not piece: 
                    board.push(move)
                    continue
                    
                piece_char = piece.symbol().upper() 
                uci = move.uci()
                
                # B. Mirror Logic (Visual Only)
                if should_mirror:
                    uci = mirror_uci_string(uci)
                
                # C. Generate Token
                token = get_compact_token(uci, piece_char)
                game_tokens.append(token)
                
                # D. Update REAL board
                board.push(move)
            
            # Write sequence to file (Space separated, One game per line)
            # NO Result token appended.
            out_f.write(" ".join(game_tokens) + "\n")
            all_tokens.update(game_tokens)
            pbar.update(1)

    # --- Save Vocab ---
    # We sort by frequency to keep common tokens at lower indices
    vocab = {t: i for i, (t, _) in enumerate(all_tokens.most_common())}
    
    # Add special tokens for the Transformer
    specials = ["[PAD]", "[BOS]"] # BOS for Start of Sequence if needed later
    for s in specials:
        if s not in vocab: 
            # Insert at the beginning or end. 
            # Re-creating dict to put specials at 0 and 1 is usually safer for PyTorch
            new_vocab = {s: i for i, s in enumerate(specials)}
            shift = len(specials)
            for t, idx in vocab.items():
                new_vocab[t] = idx + shift
            vocab = new_vocab
            break
            
    with open("vocab.pkl", "wb") as f:
        pickle.dump(vocab, f)
        
    print(f"\nDone. Vocab Size: {len(vocab)}")
    print(f"Example Tokens: {list(vocab.keys())[5:15]}") # Skip specials to see real moves

In [ ]:
process_pgn('archives/Bobby Fischer-black.pgn', 'input.txt')

In [11]:
X = np.load("data/dataset_X_sp.npy")
Y = np.load("data/dataset_Y_sp.npy")
# X = X[np.abs(Y) < 0.98]
# Y = Y[np.abs(Y) < 0.98]
X = X[~np.isnan(Y)]
Y = Y[~np.isnan(Y)]
print(X.shape)
print(Y.shape)

print(np.mean(Y), np.std(Y))
# for i in range(1, 15):
#     X_new = np.load(f"data/dataset_X_sp ({i}).npy")
#     Y_new = np.load(f"data/dataset_Y_sp ({i}).npy")

#     X = np.concatenate((X, X_new), axis=0)
#     Y = np.concatenate((Y, Y_new), axis=0)

# np.save("data/dataset_X_sp.npy", X)
# np.save("data/dataset_Y_sp.npy", Y)

# for i in range(100):
#     rand_idx = np.random.randint(0, len(X))
#     print(Y[i])

(3315940, 64)
(3315940,)
0.11451745 0.7358143


In [ ]:
x_path = 'data/dataset_X_mega.npy'
y_path = 'data/dataset_Y_mega.npy'
X = np.load(x_path)
Y = np.load(y_path)
X = X[~np.isnan(Y)]
Y = Y[~np.isnan(Y)]
 


In [ ]:
import numpy as np
import chess
import chess.svg
from IPython.display import display
from value_transformer.inference import ChessEvaluator
from value_transformer.approximate import approximate_cp
# Assuming files are in the same directory as the notebook

print(f"Loading {x_path} and {y_path}...")
try:
    print(f"Loaded {len(X)} samples.")

    # 2. Select Random Sample
    idx = np.random.randint(0, len(X))
    tokens = X[idx]
    value = Y[idx]

    # 3. Reconstruct Board
    # Mapping must match retrain.py
    int_to_symbol = {
        0: None, 1: 'P', 2: 'N', 3: 'B', 4: 'R', 5: 'Q', 6: 'K',
        7: 'p', 8: 'n', 9: 'b', 10: 'r', 11: 'q', 12: 'k'
    }

    board = chess.Board(None) # Empty board
    board.clear()


    # Tokens are stored: Rank 7 -> 0, File 0 -> 7
    token_idx = 0
    for rank in range(7, -1, -1):
        for file in range(8):
            t = tokens[token_idx]
            token_idx += 1
            if t != 0:
                symbol = int_to_symbol.get(t)
                if symbol:
                    piece = chess.Piece.from_symbol(symbol)
                    board.set_piece_at(chess.square(file, rank), piece)
    

    ValueTransformer = ChessEvaluator('value_transformer/checkpoints/mini_value_5o1.pt')
    val_cp = ValueTransformer.evaluate(board.fen())
    approx_cp = np.tanh(approximate_cp(board) / 400)

    # Stockfish Eval
    import chess.engine
    try:
        sf_path = "/opt/homebrew/bin/stockfish" # Found in previous step
        with chess.engine.SimpleEngine.popen_uci(sf_path) as sf:
            result = sf.analyse(board, chess.engine.Limit(time=0.1))
            score = result["score"].white()
            if score.is_mate():
                print(f"Stockfish Eval (White): Mate {score.mate()}")
            else:
                print(f"Stockfish Eval (White): {score.score()}")
    except Exception as e:
        print(f"Stockfish Initialization Error: {e}")
    
    # 4. Display
    # Set turn based on whose move it implies (usually input is board state before move)
    # Note: The simple tokenization doesn't store turn, castling rights, or en passant.
    # We default to White to move for visualization or just show the pieces.
    
    print(f"Sample Index: {idx}")
    print(f"Stored Value (tanh scaled): {value:.4f}")
    print(f"val cp {val_cp}", f"approx cp {approx_cp}")

    # Optional: Convert back to Approx Centipawns
    # value = tanh(cp / 400)  =>  cp = atanh(value) * 400
    try:
        est_cp = np.arctanh(value) * 400
        print(f"Estimated CP: {est_cp:.2f}")
        
    except:
        print("Estimated CP: +/- Inf (value saturated)")
        
    display(chess.svg.board(board=board, size=400))

except FileNotFoundError:
    print(f"Error: Could not find {x_path} or {y_path} in current directory.")
except Exception as e:
    print(f"An error occurred: {e}")